In [ ]:
import pyrte_rrtmgp as pyr
import xarray as xr
import numpy as np

from pyrte_rrtmgp.data_types import CloudOpticsFiles, GasOpticsFiles, AllSkyExampleFiles
from pyrte_rrtmgp.utils import compute_profiles, compute_clouds, load_rrtmgp_file


# Create an RCEMIP atmosphere with clouds

In [ ]:
#
# TK: particle sizes (which are taken from the cloud optics, awkardly)
#
def create_allsky_test_problem(ncol=24, nlay=72, sfc_T=300):
    ```
    Return an xarray dataset containing the fields needed to compute gas and cloud optics. 

    Default values for problem size and surface temperature can be compared to stored answers in RRTMGP data repo
    ```
    #
    # Create atmospheric profiles and gas concentrations
    #
    atmosphere = pyr.utils.compute_profiles(scf_T, ncol, nlay)
    #
    #  Add other gas values
    #
    gas_values = {
        "co2": 348e-6,
        "ch4": 1650e-9,
        "n2o": 306e-9,
        "n2": 0.7808,
        "o2": 0.2095,
        "co": 0.0,
    }
    #
    # Should be able to provide scalar values
    #
    for gas_name, value in gas_values.items():
        atmosphere[gas_name] = xr.DataArray(
            value,
            dims=["site", "layer"],
            coords={"site": range(ncol), "layer": range(nlay)},
        )
    #
    # Calculate cloud physical properties and merge into the atmosphere dataset
    #
    cloud_properties = pyr.utils.compute_clouds(
        cloud_optics_lw, atmosphere["pres_layer"], atmosphere["temp_layer"]
    )
    atmosphere = atmosphere.merge(cloud_properties)

    return(atmosphere)


# Longwave example

In [ ]:
gas_optics_lw = pyr.rrtmgp_gas_optics.load_gas_optics(GasOpticsFiles.LW_G256)
cld_optics_lw = pyr.rrtmgp_cloud_optics.load_cloud_optics(CloudOpticsFiles.LW_BND)

#
# Physical properties of the atmosphere
#
atmosphere = create_allsky_test_problem()

#
# Optical properties 
#
clouds_optical_props = cloud_optics_lw.compute_cloud_optics(
    atmosphere, problem_type="absorption",  add_to_input=False, 
)
optical_props = gas_optics_lw.compute_gas_optics(
    atmosphere, problem_type="absorption", add_to_input=False, 
)
clouds_optical_props.add_to(optical_props)

#
# Boundary conditions 
#
optical_props["surface_emissivity"] = 0.98

#
# Solution
#
fluxes = pyr.rte_solver.rte_solve(optical_props, add_to_input=False)

#
# Load reference data and verify results
#
ref_data = load_rrtmgp_file(AllSkyExampleFiles.LW_NO_AEROSOL)
assert np.isclose(fluxes["lw_flux_up"], ref_data["lw_flux_up"].T, atol=1e-7).all()
assert np.isclose(fluxes["lw_flux_down"], ref_data["lw_flux_dn"].T, atol=1e-7).all()